## 1. 安裝與編譯（僅首次）

In [ ]:
# Import tools used for compilation, paths and GPU inspection.
# 匯入編譯、路徑與 GPU 檢查工具。
import shutil
# Access process management.
# 使用程序管理工具。
import subprocess
# Resolve current Python environment.
# 取得目前的 Python 執行環境。
import sys
# Manage local directories.
# 管理本機資料夾。
from pathlib import Path

# Install only the download and HTTP dependencies.
# 僅安裝模型下載與 HTTP 所需套件。
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'huggingface_hub', 'requests'], check=True)

# Set Kaggle paths for the compiled engine.
# 設定 Kaggle 編譯引擎的路徑。
ROOT = Path('/kaggle/working')
# Keep source code reusable during the active session.
# 在目前 Session 中重複利用原始碼。
PROJECT = ROOT / 'yue2.cpp'
# Select the Kaggle CUDA compiler.
# 指定 Kaggle 的 CUDA 編譯器。
NVCC = shutil.which('nvcc') or '/usr/local/cuda/bin/nvcc'

# Require two NVIDIA GPUs and the CUDA compiler.
# 確認雙 GPU 與 CUDA 編譯器可用。
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True, check=True)
# Avoid silently using just one GPU.
# 避免只有一張 GPU 時仍繼續執行。
assert len(gpu.stdout.strip().splitlines()) == 2, '請在 Kaggle 啟用 GPU T4 ×2。'
# Confirm nvcc can compile the CUDA backend.
# 確認 nvcc 可以用來編譯 CUDA 後端。
assert Path(NVCC).is_file(), '找不到 nvcc，請確認 Kaggle CUDA Toolkit。'
# Report the two GPU devices.
# 顯示兩張 GPU 裝置。
print(gpu.stdout)

# Download the official C++ engine only when missing.
# 只有尚未下載時才取得官方 C++ 引擎。
if not (PROJECT / '.git').exists():
    # Clone the engine and its GGML submodule.
    # 下載引擎及 GGML 子模組。
    subprocess.run(['git', 'clone', '--depth', '1', '--recurse-submodules', 'https://github.com/ServeurpersoCom/yue2.cpp.git', str(PROJECT)], check=True)
# Repair an incomplete submodule checkout without recloning.
# 補齊不完整的子模組，不重複下載專案。
subprocess.run(['git', '-C', str(PROJECT), 'submodule', 'update', '--init', '--recursive'], check=True)

# Compile for the T4's SM 7.5 architecture.
# 針對 T4 的 SM 7.5 架構編譯。
subprocess.run(['cmake', '-S', str(PROJECT), '-B', str(PROJECT / 'build'), '-DGGML_CUDA=ON', '-DCMAKE_BUILD_TYPE=Release', '-DCMAKE_CUDA_ARCHITECTURES=75', f'-DCMAKE_CUDA_COMPILER={NVCC}'], check=True)
# Build the persistent server, not the one-shot CLI.
# 編譯可常駐的 Server，而不是單次執行 CLI。
subprocess.run(['cmake', '--build', str(PROJECT / 'build'), '--target', 'yue-server', '--parallel', '2'], check=True)
# Save the executable location for the following cells.
# 儲存下一格會用到的執行檔路徑。
SERVER = PROJECT / 'build/yue-server'
# Stop early if the build did not produce the server.
# 若沒有成功編譯 Server，立即停止。
assert SERVER.is_file(), '找不到 yue-server，請檢查編譯紀錄。'

## 2. 下載 BF16 模型（僅首次）

In [ ]:
# Import the Hugging Face file downloader.
# 匯入 Hugging Face 模型下載工具。
from huggingface_hub import hf_hub_download

# Use one shared disk copy for both GPU servers.
# 讓兩個 GPU Server 共用同一份磁碟權重。
MODEL_DIR = ROOT / 'yue2_models'
# Create the model directory if necessary.
# 必要時建立模型資料夾。
MODEL_DIR.mkdir(parents=True, exist_ok=True)
# Select the source of the unquantized YuE2 model.
# 指定未量化 YuE2 模型的來源。
REPO = 'Serveurperso/YuE2-GGUF'

# Download the full-precision BF16 generation backbone.
# 下載原始精度 BF16 主模型。
BACKBONE = Path(hf_hub_download(repo_id=REPO, filename='YuE2-3B-BF16.gguf', local_dir=str(MODEL_DIR)))
# Download the official F32 VAE decoder.
# 下載官方 F32 音訊 VAE 解碼器。
VAE = Path(hf_hub_download(repo_id=REPO, filename='YuE2-Vae-F32.gguf', local_dir=str(MODEL_DIR)))
# Reject incomplete downloads.
# 排除不完整的模型下載。
assert BACKBONE.stat().st_size > 6 * 1024**3 and VAE.stat().st_size > 400 * 1024**2, '模型檔案大小異常。'
# Report the actual downloaded filenames.
# 顯示下載完成的模型名稱。
print('Models:', BACKBONE.name, '+', VAE.name)

## 3. 啟動雙 GPU 常駐服務（僅首次）

In [ ]:
# Import HTTP requests and time monitoring.
# 匯入 HTTP 與計時工具。
import requests
# Control service initialization timeouts.
# 控制服務初始化的等待限制。
import time

# Assign one independent server to each GPU.
# 每張 GPU 分配一個獨立 Server。
PORTS = {0: 8091, 1: 8092}
# Keep process handles when rerunning the cell.
# 重複執行此格時保留既有程序。
SERVER_PROCS = globals().get('SERVER_PROCS', {})

# Initialize both GPU services without downloading the model twice.
# 啟動兩個 GPU 服務，不重複下載模型。
for gpu_id, port in PORTS.items():
    # Restrict each API to the local Kaggle notebook.
    # 只讓目前 Kaggle Notebook 連線至服務。
    address = f'http://127.0.0.1:{port}'
    # Reuse an existing healthy process.
    # 如果程序仍正常運作，就沿用它。
    process = SERVER_PROCS.get(gpu_id)
    # Check whether the previous process is available.
    # 檢查既有程序是否仍可使用。
    if process is not None and process.poll() is None:
        # Reuse the process rather than reloading the model.
        # 沿用程序，避免模型重複載入。
        requests.get(address + '/health', timeout=5).raise_for_status()
        # Confirm the reused GPU service.
        # 顯示重複使用的 GPU 服務。
        print(f'GPU {gpu_id}: server reused on port {port}')
        # Start the next GPU service.
        # 接著檢查另一張 GPU。
        continue
    # Preserve diagnostics from model-loading failures.
    # 保留模型載入失敗時的診斷紀錄。
    log_path = ROOT / f'yue2_gpu{gpu_id}.log'
    # Configure the official resident-model server mode.
    # 啟用官方模型常駐選項。
    command = [str(SERVER), '--model', str(BACKBONE), '--vae', str(VAE), '--device', f'CUDA{gpu_id}', '--host', '127.0.0.1', '--port', str(port), '--keep-loaded', '--max-seq', '8192', '--vae-core', '256']
    # Launch the server independently of the Jupyter cell.
    # 讓服務在 Jupyter Cell 完成後持續運作。
    with log_path.open('a', encoding='utf-8') as log:
        # Store a process handle for reuse.
        # 保留供後續沿用的程序控制項。
        process = subprocess.Popen(command, cwd=PROJECT, stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
    # Retain the process handle.
    # 保存目前 GPU 的程序控制項。
    SERVER_PROCS[gpu_id] = process
    # Wait until the official health endpoint responds.
    # 等待官方健康檢查 API 回應。
    for attempt in range(90):
        # Catch the startup error before a generation begins.
        # 在開始生成前先排除啟動失敗的情況。
        assert process.poll() is None, f'GPU {gpu_id} 服務已結束，請查看 {log_path}'
        # Try the health endpoint even while the server is starting.
        # 即使服務尚在啟動，也持續檢查健康狀態。
        try:
            # Confirm that the server accepts API requests.
            # 確認服務已接受 API 請求。
            response = requests.get(address + '/health', timeout=2)
            # Stop waiting after a successful response.
            # 健康檢查成功即可停止等待。
            if response.ok:
                # Leave the startup-check loop.
                # 結束服務啟動檢查。
                break
        # Treat temporary startup connection errors as normal.
        # 服務啟動過程中的暫時連線錯誤屬正常情況。
        except requests.RequestException:
            # Recheck after a short delay.
            # 稍後重新檢查。
            pass
        # Avoid continuous polling during initialization.
        # 避免啟動期間過度頻繁查詢。
        time.sleep(1)
    # Refuse to generate if the service never starts.
    # 若服務無法啟動，則不進行生成。
    else:
        # Report the relevant server log for diagnosis.
        # 顯示有問題的服務紀錄。
        raise RuntimeError(f'GPU {gpu_id} 初始化失敗：{log_path.read_text(errors="replace")[-2500:]}')
    # Report a ready server without claiming models are preloaded.
    # 回報服務就緒，但不假設權重已完成載入。
    print(f'GPU {gpu_id}: service ready at {address}')

# Explain when the resident model first enters VRAM.
# 說明模型何時首次載入 VRAM。
print('BF16 weights will load on each GPU during its first generation, then stay resident.')

## 4. 生成設定

In [ ]:
# Set the music genre and vocal characteristics.
# 設定音樂曲風與人聲特性。
STYLE = 'Mandarin pop, warm female vocal, acoustic guitar, soft piano, ballad, 90 BPM'

# Set the song lyrics.
# 設定歌曲歌詞。
LYRICS = """[Verse]
微風吹過熟悉的街道
晚霞落在回家的轉角

[Chorus]
讓心裡的光慢慢閃耀
明天還有新的美好"""

# Set the requested song duration in seconds.
# 設定生成音樂長度（秒）。
DURATION = 30
# Set the number of acoustic refinement steps.
# 設定聲學細化步數。
STEPS = 32
# Generate an ABC score before rendering the audio.
# 先生成 ABC 樂譜再合成音樂。
COT = 'full'
# Use fresh random seeds for each song.
# 每首歌曲使用不同的隨機種子。
LM_SEED, SEED = -1, -1
# Generate one variation on each selected GPU.
# 每張指定的 GPU 各生成一個版本。
GPU_IDS = (0, 1)
# Set the output name; existing MP3 files will not be overwritten.
# 設定輸出檔名，不覆蓋既有 MP3。
OUTPUT_NAME = 'yue2_bf16.mp3'

## 5. 生成音樂（可重複執行）

In [ ]:
# Import JSON parsing for the returned song metadata.
# 匯入 JSON 解析工具，讀取歌曲資訊。
import json
# Decode the server's multipart audio response.
# 解碼 Server 回傳的 multipart 音訊資料。
from email.parser import BytesParser
# Preserve binary audio exactly without content conversions.
# 原樣保存音訊二進位資料。
from email import policy
# Run two independent GPU jobs concurrently.
# 同時執行兩張 GPU 各自的生成工作。
from concurrent.futures import ThreadPoolExecutor
# Access audio downloads and request status.
# 使用音訊下載與任務狀態 API。
import requests
# Poll jobs without unnecessary CPU activity.
# 以適當間隔查詢任務，避免過度占用 CPU。
import time

# Verify the selected GPU numbers.
# 確認選用的 GPU 編號合法。
assert GPU_IDS and set(GPU_IDS).issubset(PORTS), 'GPU_IDS 只能包含 0 或 1。'
# Create one shared output directory.
# 建立共用音樂輸出資料夾。
OUTPUT_DIR = ROOT / 'yue2_outputs'
# Reuse the output directory for subsequent songs.
# 重複生成時沿用同一個資料夾。
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Construct the official YuE2 generation request.
# 建立官方 YuE2 音樂生成請求。
request_data = {'style': STYLE, 'lyrics': LYRICS, 'cot': COT, 'duration': DURATION, 'steps': STEPS, 'lm_seed': LM_SEED, 'seed': SEED, 'lm_batch_size': 1, 'synth_batch_size': 1, 'output_format': 'mp3', 'mp3_bitrate': 192}

# Generate a separate composition on the chosen GPU.
# 在指定 GPU 生成一首獨立歌曲。
def generate_on_gpu(gpu_id):
    # Identify the resident server assigned to this GPU.
    # 取得這張 GPU 對應的常駐服務。
    base = f'http://127.0.0.1:{PORTS[gpu_id]}'
    # Send the request without rebuilding or reloading the model.
    # 直接送出生成請求，不重新編譯或初始化模型。
    submitted = requests.post(base + '/synth', json=request_data, timeout=60)
    # Stop if the server rejects the input.
    # 若 Server 拒絕請求則停止。
    submitted.raise_for_status()
    # Read the job identifier returned by the server.
    # 取得服務回傳的工作編號。
    job_id = submitted.json()['id']
    # Show the GPU assignment and job identifier.
    # 顯示任務使用的 GPU 與編號。
    print(f'GPU {gpu_id}: generating, job={job_id}', flush=True)
    # Keep a deadline for extremely long or stalled generations.
    # 為執行過久或停滯的生成設定截止時間。
    deadline = time.monotonic() + 3600
    # Poll until this independent job has finished.
    # 定期檢查本張 GPU 的生成是否完成。
    while time.monotonic() < deadline:
        # Retrieve the current job state.
        # 取得目前生成工作狀態。
        response = requests.get(base + '/job', params={'id': job_id}, timeout=30)
        # Surface server errors immediately.
        # 立即回報服務錯誤。
        response.raise_for_status()
        # Decode the current job state.
        # 解析工作目前狀態。
        status = response.json()
        # Exit polling when an audio result is ready.
        # 音訊完成時結束查詢。
        if status.get('status') == 'done':
            # Continue to retrieve the completed result.
            # 接著讀取已完成的音訊。
            break
        # Preserve the server's error message on failure.
        # 生成失敗時保留原始錯誤內容。
        if status.get('status') in ('failed', 'cancelled'):
            # Report the failed job's status details.
            # 回報失敗任務的細節。
            raise RuntimeError(f'GPU {gpu_id}: {status}')
        # Avoid overloading the local job-status API.
        # 避免頻繁查詢工作狀態 API。
        time.sleep(5)
    # Stop a generation that exceeded the safety time limit.
    # 超過等待時間上限時停止。
    else:
        # Preserve the job identifier for manual inspection.
        # 保留工作編號以供後續檢查。
        raise TimeoutError(f'GPU {gpu_id}: job {job_id} still running after one hour')

    # Request the audio and replay metadata as multipart data.
    # 使用 multipart 格式取得音訊與生成資訊。
    result = requests.get(base + '/job', params={'id': job_id, 'result': 1}, timeout=120)
    # Raise an explicit error if the result is unavailable.
    # 若輸出尚無法取得，立即回報錯誤。
    result.raise_for_status()
    # Create a complete MIME document for the standard parser.
    # 將回應組合成標準 MIME 文件供解析。
    mime = b'MIME-Version: 1.0\r\nContent-Type: ' + result.headers['Content-Type'].encode() + b'\r\n\r\n' + result.content
    # Parse the response without misidentifying metadata as MP3.
    # 解析回應，避免將資訊文字誤存為 MP3。
    message = BytesParser(policy=policy.default).parsebytes(mime)
    # Require the official multipart/mixed response format.
    # 確認回應符合官方 multipart/mixed 格式。
    assert message.is_multipart(), f'GPU {gpu_id}: invalid audio response'
    # Separate the audio part from the replay metadata.
    # 從回應中分離音訊及生成參數。
    parts = list(message.iter_parts())
    # Locate the encoded MP3 audio part.
    # 找出真正的 MP3 音訊資料。
    audio = next((part.get_payload(decode=True) for part in parts if part.get_content_type() == 'audio/mpeg'), None)
    # Reject an absent or implausibly small audio stream.
    # 排除缺失或明顯不完整的音訊。
    assert audio and len(audio) > 4096, f'GPU {gpu_id}: empty MP3 result'
    # Avoid overwriting songs from previous generations.
    # 避免覆蓋先前生成的歌曲。
    stem = Path(OUTPUT_NAME).stem or 'yue2'
    # Pick the first available song filename for this GPU.
    # 為這張 GPU 選出尚未使用的歌曲檔名。
    index = 1
    # Find a filename not used by any earlier generation.
    # 找出不會與先前生成結果重複的檔名。
    while (OUTPUT_DIR / f'{stem}_gpu{gpu_id}_{index:03d}.mp3').exists():
        # Advance to the next available suffix.
        # 逐一尋找未使用的編號。
        index += 1
    # Resolve the final MP3 path.
    # 取得最終 MP3 儲存路徑。
    output = OUTPUT_DIR / f'{stem}_gpu{gpu_id}_{index:03d}.mp3'
    # Save the decoded MP3 bytes to disk.
    # 將解碼後的 MP3 資料儲存到磁碟。
    output.write_bytes(audio)
    # Save optional replay information and ABC score.
    # 儲存選用的重播參數及 ABC 樂譜。
    for part in parts:
        # Process only the response's JSON metadata part.
        # 僅處理回應中的 JSON 資訊。
        if part.get_content_type() == 'application/json':
            # Decode the official generation replay request.
            # 解析官方生成重播資訊。
            replay = json.loads(part.get_payload(decode=True).decode('utf-8'))
            # Save reproducibility information beside the audio.
            # 將可重現的生成參數儲存在音訊旁。
            output.with_suffix('.json').write_text(json.dumps(replay, ensure_ascii=False, indent=2), encoding='utf-8')
            # Persist the generated score if the server supplied one.
            # 如果服務有回傳樂譜，則另外儲存。
            if replay.get('abc'):
                # Store the ABC score as a standalone text file.
                # 將 ABC 樂譜儲存為獨立文字檔。
                output.with_suffix('.abc').write_text(replay['abc'], encoding='utf-8')
            # Avoid processing multiple replay metadata parts.
            # 不重複處理多份生成參數。
            break
    # Report the completed output for this GPU.
    # 回報這張 GPU 已完成的音樂。
    print(f'GPU {gpu_id}: saved {output.name}', flush=True)
    # Return the path for notebook playback.
    # 回傳路徑供 Notebook 試聽。
    return output

# Execute the two independent GPU jobs in parallel.
# 讓兩張 GPU 同時生成各自的歌曲。
with ThreadPoolExecutor(max_workers=len(GPU_IDS)) as pool:
    # Keep the resulting paths for playback and download.
    # 儲存生成結果的路徑供試聽下載。
    OUTPUTS = list(pool.map(generate_on_gpu, GPU_IDS))

## 6. 試聽與下載

In [ ]:
# Import audio playback and notebook links.
# 匯入音訊試聽與檔案下載連結。
from IPython.display import Audio, FileLink, display

# Preview each independent GPU's generated song.
# 依序試聽兩張 GPU 各自生成的歌曲。
for output in OUTPUTS:
    # Show the local output filename.
    # 顯示輸出的音樂檔名。
    print(output.name)
    # Play the generated MP3 directly in Kaggle.
    # 直接在 Kaggle 試聽生成的 MP3。
    display(Audio(filename=str(output)))
    # Provide a link for downloading the MP3.
    # 提供 MP3 檔案下載連結。
    display(FileLink(str(output)))